In [31]:
from transformers import AutoConfig, AutoModelForCausalLM
import torch
import torch.nn as nn
import copy

cfg = AutoConfig.from_pretrained("ibm-granite/granite-3.0-1b-a400m-base")
cfg.num_hidden_layers = 2
cfg.num_local_experts = 4
cfg.num_experts_per_tok = 2
torch.manual_seed(42)
tiny = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.float32).eval()   # random weights, loads instantly on CPU

In [32]:
print(f"tiny model has {sum(p.numel() for p in tiny.parameters())/1e6:.2f}M parameters")
print()
print("Model Architecture:")
print(tiny)

tiny model has 69.22M parameters

Model Architecture:
GraniteMoeForCausalLM(
  (model): GraniteMoeModel(
    (embed_tokens): Embedding(49152, 1024, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x GraniteMoeDecoderLayer(
        (self_attn): GraniteMoeAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (input_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (block_sparse_moe): GraniteMoeMoE(
          (router): GraniteMoeTopKRouter()
          (experts): GraniteMoeExperts(
            (act_fn): SiLUActivation()
          )
        )
      )
    )
    (norm): GraniteMoeRMSNorm((1024,), eps=1e-06)
    (rotary_emb): GraniteM

In [33]:
torch.manual_seed(42)
tiny_new = copy.deepcopy(tiny)
print(f"tiny_new model has {sum(p.numel() for p in tiny_new.parameters())/1e6:.2f}M parameters")
print(f"Embedding weights are equal: {torch.equal(tiny.model.embed_tokens.weight, tiny_new.model.embed_tokens.weight)}")  # False
print()
print("Model Architecture:")
print(tiny_new)

tiny_new model has 69.22M parameters
Embedding weights are equal: True

Model Architecture:
GraniteMoeForCausalLM(
  (model): GraniteMoeModel(
    (embed_tokens): Embedding(49152, 1024, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x GraniteMoeDecoderLayer(
        (self_attn): GraniteMoeAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (input_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (block_sparse_moe): GraniteMoeMoE(
          (router): GraniteMoeTopKRouter()
          (experts): GraniteMoeExperts(
            (act_fn): SiLUActivation()
          )
        )
      )
    )
    (norm): GraniteMoeRMSNorm((1024,),

In [34]:
class Expert(nn.Module):

    def __init__(self, hidden, inter, act_fn):
        super().__init__()
        self.gate = nn.Linear(hidden, inter, bias=False)
        self.up   = nn.Linear(hidden, inter, bias=False)
        self.down = nn.Linear(inter, hidden, bias=False)
        self.act_fn = act_fn

    def forward(self, x):
        return self.down(self.act_fn(self.gate(x)) * self.up(x))


class ModularExperts(nn.Module):

    def __init__(self, n_experts, hidden, inter, act_fn=nn.SiLU()):
        super().__init__()
        self.experts = nn.ModuleList(Expert(hidden, inter, act_fn) for _ in range(n_experts))


    @classmethod
    def from_fused(cls, fused):
        W_gu, W_dn = fused.gate_up_proj.data, fused.down_proj.data
        E, H, I = W_dn.shape
        obj = cls(E, H, I, fused.act_fn).to(W_gu)
        with torch.no_grad():
            for e, ex in enumerate(obj.experts):
                ex.gate.weight.copy_(W_gu[e, :I])
                ex.up.weight.copy_(W_gu[e, I:])
                ex.down.weight.copy_(W_dn[e])
        return obj


    @property
    def num_experts(self):
        return len(self.experts)


    def forward(self, hidden_states, top_k_index, top_k_weights):
        out = torch.zeros_like(hidden_states)
        for e, expert in enumerate(self.experts):
            token_idx, slot = torch.where(top_k_index == e)   # indices >= len(experts) never match
            if token_idx.numel() == 0:
                continue
            y = expert(hidden_states[token_idx]) * top_k_weights[token_idx, slot, None]
            out.index_add_(0, token_idx, y.to(out.dtype))
        return out


for layer in tiny_new.model.layers:
    layer.block_sparse_moe.experts = ModularExperts.from_fused(layer.block_sparse_moe.experts)


print(f"tiny_new model has {sum(p.numel() for p in tiny_new.parameters())/1e6:.2f}M parameters")
print()
print("Model Architecture:")
print(tiny_new)

tiny_new model has 69.22M parameters

Model Architecture:
GraniteMoeForCausalLM(
  (model): GraniteMoeModel(
    (embed_tokens): Embedding(49152, 1024, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x GraniteMoeDecoderLayer(
        (self_attn): GraniteMoeAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (input_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (block_sparse_moe): GraniteMoeMoE(
          (router): GraniteMoeTopKRouter()
          (experts): ModularExperts(
            (experts): ModuleList(
              (0-3): 4 x Expert(
                (gate): Linear(in_features=1024, out_features=512, bias=False)
   

In [35]:
@torch.no_grad()
def check_weights(old, new):
    """Every expert's weights must be exact copies of the fused slices."""
    W_gu, W_dn = old.gate_up_proj, old.down_proj        # [E, 2I, H], [E, H, I]
    E, H, I = W_dn.shape

    assert len(new.experts) == E, f"expert count: {len(new.experts)} vs {E}"
    for e, ex in enumerate(new.experts):
        assert torch.equal(ex.gate.weight, W_gu[e, :I]), f"expert {e}: gate mismatch"
        assert torch.equal(ex.up.weight,   W_gu[e, I:]), f"expert {e}: up mismatch"
        assert torch.equal(ex.down.weight, W_dn[e]),     f"expert {e}: down mismatch"
        # must be separate memory, not views into the fused tensor
        assert ex.gate.weight.data_ptr() != W_gu[e, :I].data_ptr(), f"expert {e}: gate shares memory"
    print(f"check_weights OK: {E} experts, hidden {H}, intermediate {I}")


@torch.no_grad()
def check_module(old, new, k=None, n_tokens=256, drop_frac=0, seed=0):
    """Same input + same routing must give the same output."""
    E, H, _ = old.down_proj.shape
    print(f"check_module: {E} experts, hidden {H}, k={k}, n_tokens={n_tokens}, drop_frac={drop_frac}")
    k = k or min(8, E)
    dev, dt = old.down_proj.device, old.down_proj.dtype
    g = torch.Generator(device="cpu").manual_seed(seed)

    x = torch.randn(n_tokens, H, generator=g).to(dev, dt)
    # k distinct experts per token
    idx = torch.rand(n_tokens, E, generator=g).argsort(-1)[:, :k].to(dev)
    # some slots marked "no expert" (index E), as the library does; both must skip them
    drop = torch.rand(n_tokens, k, generator=g) < drop_frac
    idx[drop.to(dev)] = E
    w = torch.rand(n_tokens, k, generator=g).softmax(-1).to(dev, dt)

    out_old = old(x, idx, w)
    out_new = new(x, idx, w)
    torch.testing.assert_close(out_new, out_old)

    used = torch.unique(idx[idx < E]).numel()
    diff = (out_new - out_old).abs().max().item()
    print(f"check_module OK: {used}/{E} experts used, k={k}, max |Δ| = {diff:.2e}")

In [36]:
for l_old, l_new in zip(tiny.model.layers, tiny_new.model.layers):
    old, new = l_old.block_sparse_moe.experts, l_new.block_sparse_moe.experts
    check_weights(old, new)
    check_module(old, new)

check_weights OK: 4 experts, hidden 1024, intermediate 512
check_module: 4 experts, hidden 1024, k=None, n_tokens=256, drop_frac=0


check_module OK: 4/4 experts used, k=4, max |Δ| = 4.47e-08
check_weights OK: 4 experts, hidden 1024, intermediate 512
check_module: 4 experts, hidden 1024, k=None, n_tokens=256, drop_frac=0
check_module OK: 4/4 experts used, k=4, max |Δ| = 4.47e-08


### Extend the experts

In [37]:
router = tiny_new.model.layers[0].block_sparse_moe.router
for n, p in router.named_parameters():
    print(n, tuple(p.shape))                # expect one weight of shape (4, 1024)
print({k: v for k, v in vars(router).items() if not k.startswith("_")})   # e.g. top_k, num_experts

weight (4, 1024)
{'training': False, 'num_experts': 4, 'top_k': 2}


In [38]:
print(tiny_new)
print()
print(tiny.model.layers[0].block_sparse_moe.router)

GraniteMoeForCausalLM(
  (model): GraniteMoeModel(
    (embed_tokens): Embedding(49152, 1024, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x GraniteMoeDecoderLayer(
        (self_attn): GraniteMoeAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (input_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): GraniteMoeRMSNorm((1024,), eps=1e-06)
        (block_sparse_moe): GraniteMoeMoE(
          (router): GraniteMoeTopKRouter()
          (experts): ModularExperts(
            (experts): ModuleList(
              (0-3): 4 x Expert(
                (gate): Linear(in_features=1024, out_features=512, bias=False)
                (up): Linear(in_features=1024, out_features=5

In [39]:
@torch.no_grad()
def add_experts(model, n_new, num_experts_per_tok_new=None,):

    moes = [l.block_sparse_moe for l in model.model.layers]
    assert all(isinstance(m.experts, ModularExperts) for m in moes), "run modularize() first"
    ex0 = moes[0].experts.experts[0]
    hidden, inter, act_fn = ex0.gate.in_features, ex0.up.out_features, ex0.act_fn
    dev, dt = ex0.gate.weight.device, ex0.gate.weight.dtype
    print(f"Adding {n_new} experts to model with {len(moes[0].experts.experts)} existing experts, hidden {hidden}, intermediate {inter}, act_fn {act_fn}")
    for layer in model.model.layers:
        layer.block_sparse_moe.experts.experts.extend([Expert(hidden, inter, act_fn).to(device=dev, dtype=dt) for _ in range(n_new)])
        layer.block_sparse_moe.router.num_experts += n_new
        if num_experts_per_tok_new is not None:
            layer.block_sparse_moe.router.top_k = num_experts_per_tok_new
        W = layer.block_sparse_moe.router.weight                                            # (E, H)
        new_rows = torch.randn(n_new, W.shape[1], device=W.device, dtype=W.dtype) * W.std()
        layer.block_sparse_moe.router.weight = nn.Parameter(torch.cat([W.data, new_rows], dim=0))   # (E + n_new, H)
    model.config.num_local_experts += n_new
    if num_experts_per_tok_new is not None:
        model.config.num_experts_per_tok = num_experts_per_tok_new

In [40]:
tiny_extended = copy.deepcopy(tiny_new)
add_experts(tiny_extended, n_new=2, num_experts_per_tok_new=3)

Adding 2 experts to model with 4 existing experts, hidden 1024, intermediate 512, act_fn SiLUActivation()


### Test the router

In [41]:
@torch.no_grad()
def expert_usage(model, batches):
    layers = model.model.layers
    counts = [torch.zeros(len(l.block_sparse_moe.experts.experts), dtype=torch.long) for l in layers]
    hooks = []
    for i, l in enumerate(layers):
        def hook(mod, args, i=i):
            idx = args[1].flatten().cpu()                   # top_k_index
            idx = idx[idx < len(counts[i])]
            counts[i] += torch.bincount(idx, minlength=len(counts[i]))
        hooks.append(l.block_sparse_moe.experts.register_forward_pre_hook(hook))
    try:
        for b in batches:
            model(**b)
    finally:
        for h in hooks:
            h.remove()
    return counts

In [42]:
torch.manual_seed(0)
input_ids = torch.randint(0, tiny.config.vocab_size, (4, 128))    # 4 sequences × 128 tokens = 512 tokens
batch = {"input_ids": input_ids, "attention_mask": torch.ones_like(input_ids)}

In [43]:
print(f"counts: {expert_usage(tiny_extended, [batch])}")

counts: [tensor([257, 263, 242, 263, 253, 258]), tensor([282, 262, 248, 221, 265, 258])]
